# Benchmark Datasets for Quorum-Sensing Consensus Evaluation

This notebook demonstrates the processing and standardization of benchmark datasets for evaluating hierarchical LLM consensus protocols with confidence-calibrated veto mechanisms.

## Datasets Included

1. **MMLU-Hard** - 2197 examples from 10 difficult subjects like professional_accounting and conceptual_physics
2. **GPQA** - 448 graduate-level questions from biology, physics, and chemistry
3. **TruthfulQA** - 817 questions testing truthfulness with plausible incorrect answers

## Data Schema

All datasets are standardized to a unified JSON schema with fields for:
- `id` - unique identifier
- `question` - the input question
- `choices` - array of possible answers
- `correct_answer` - index of the correct answer
- `category` - subject category
- `difficulty` - 0-1 scale
- `contestedness` - 0-1 scale (computed as 1 - baseline_accuracy)
- `metadata` - additional information

## What This Demo Does

This demo loads a mini dataset and shows how to:
1. Load the standardized data
2. Explore the dataset structure
3. Analyze difficulty and contestedness distributions
4. Display sample questions

In [ ]:
# Install dependencies
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# This notebook doesn't require special packages beyond standard library
# loguru is used in the original script but we'll use print for simplicity
print("Dependencies check complete")

In [ ]:
# Imports
import json
import os
from pathlib import Path
from collections import defaultdict

# For visualization
import matplotlib.pyplot as plt
import numpy as np

print("Imports complete")

In [ ]:
# Data loading helper - GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-d49afe-quorum-sensing-committees-a-hierarchical/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load data from GitHub URL with local fallback."""
    # Try loading from GitHub first
    try:
        import urllib.request
        print("Attempting to load data from GitHub...")
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=10) as response:
            data = json.loads(response.read().decode())
            print("Successfully loaded data from GitHub")
            return data
    except Exception as e:
        print(f"Could not load from GitHub: {e}")
    
    # Fallback to local file
    if os.path.exists("mini_demo_data.json"):
        print("Loading data from local file...")
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

print("Data loading helper defined")

In [ ]:
# Load the demo data
print("="*60)
print("Loading Demo Data")
print("="*60)

data = load_data()

print(f"\nLoaded data with {len(data['datasets'])} dataset(s)")
for dataset in data['datasets']:
    print(f"  - {dataset['dataset']}: {len(dataset['examples'])} examples")

## Data Exploration

Now let's explore the structure of our standardized dataset. The data has been processed to provide a unified schema across all three benchmark datasets (MMLU-Hard, GPQA, and TruthfulQA).

In [ ]:
# Explore dataset structure
print("="*60)
print("Dataset Structure Exploration")
print("="*60)

for dataset in data['datasets']:
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset['dataset']}")
    print(f"Number of examples: {len(dataset['examples'])}")
    
    # Examine first example structure
    if dataset['examples']:
        first_example = dataset['examples'][0]
        print(f"\nExample fields:")
        for key in first_example.keys():
            value = first_example[key]
            if isinstance(value, str) and len(value) > 60:
                print(f"  - {key}: {value[:60]}...")
            else:
                print(f"  - {key}: {value}")

## Analyze Difficulty and Contestedness

The dataset includes two important metrics:
- **Difficulty** (0-1): How challenging the question is
- **Contestedness** (0-1): Measures disagreement or ambiguity (computed as 1 - baseline_accuracy for MMLU-Hard)

Let's visualize the distribution of these metrics.

In [ ]:
# Extract and analyze difficulty and contestedness metrics
print("="*60)
print("Difficulty and Contestedness Analysis")
print("="*60)

# Collect metrics by dataset
metrics_by_dataset = {}

for dataset in data['datasets']:
    dataset_name = dataset['dataset']
    difficulties = []
    contestedness = []
    categories = defaultdict(int)
    
    for example in dataset['examples']:
        difficulties.append(example['metadata_difficulty'])
        contestedness.append(example['metadata_contestedness'])
        categories[example['metadata_category']] += 1
    
    metrics_by_dataset[dataset_name] = {
        'difficulties': difficulties,
        'contestedness': contestedness,
        'categories': categories,
        'num_examples': len(dataset['examples'])
    }

# Print summary statistics
for dataset_name, metrics in metrics_by_dataset.items():
    print(f"\n{dataset_name.upper()} Dataset:")
    print(f"  Number of examples: {metrics['num_examples']}")
    if metrics['difficulties']:
        print(f"  Difficulty - Min: {min(metrics['difficulties']):.2f}, Max: {max(metrics['difficulties']):.2f}, Mean: {np.mean(metrics['difficulties']):.2f}")
        print(f"  Contestedness - Min: {min(metrics['contestedness']):.2f}, Max: {max(metrics['contestedness']):.2f}, Mean: {np.mean(metrics['contestedness']):.2f}")
    print(f"  Categories: {dict(metrics['categories'])}")

In [ ]:
# Visualize difficulty and contestedness distributions
print("\n" + "="*60)
print("Visualization: Difficulty and Contestedness Distributions")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Difficulty distribution
all_difficulties = []
all_contestedness = []
dataset_labels = []

for dataset_name, metrics in metrics_by_dataset.items():
    all_difficulties.extend(metrics['difficulties'])
    all_contestedness.extend(metrics['contestedness'])
    dataset_labels.extend([dataset_name] * len(metrics['difficulties']))

# Difficulty histogram
axes[0].hist(all_difficulties, bins=10, alpha=0.7, color='blue', edgecolor='black')
axes[0].set_xlabel('Difficulty (0-1)')
axes[0].set_ylabel('Count')
axes[0].set_title('Difficulty Distribution')
axes[0].grid(True, alpha=0.3)

# Contestedness histogram
axes[1].hist(all_contestedness, bins=10, alpha=0.7, color='red', edgecolor='black')
axes[1].set_xlabel('Contestedness (0-1)')
axes[1].set_ylabel('Count')
axes[1].set_title('Contestedness Distribution')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Visualization complete!")

## Display Sample Questions

Let's look at some sample questions from each dataset to understand the data better.

In [ ]:
# Display sample questions from the dataset
print("="*60)
print("Sample Questions")
print("="*60)

for dataset in data['datasets']:
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset['dataset']}")
    print(f"{'='*60}")
    
    for i, example in enumerate(dataset['examples'][:3]):  # Show first 3 examples
        print(f"\n--- Example {i+1} ---")
        print(f"Category: {example['metadata_category']}")
        print(f"Difficulty: {example['metadata_difficulty']}")
        print(f"Contestedness: {example['metadata_contestedness']}")
        print(f"\nQuestion:")
        print(f"  {example['input'][:200]}..." if len(example['input']) > 200 else f"  {example['input']}")
        
        print(f"\nChoices:")
        for j, choice in enumerate(example['metadata_choices']):
            marker = " [CORRECT]" if str(j) == example['output'] else ""
            print(f"  {j}: {choice}{marker}")
        
        print(f"\nCorrect Answer Index: {example['output']}")

## Summary Statistics Table

Let's create a summary table of the dataset statistics.

In [ ]:
# Create a summary table of dataset statistics
print("="*60)
print("Dataset Summary Statistics")
print("="*60)

# Build summary data
summary_data = []

for dataset in data['datasets']:
    dataset_name = dataset['dataset']
    examples = dataset['examples']
    
    # Calculate statistics
    num_examples = len(examples)
    categories = set(ex['metadata_category'] for ex in examples)
    avg_difficulty = np.mean([ex['metadata_difficulty'] for ex in examples]) if examples else 0
    avg_contestedness = np.mean([ex['metadata_contestedness'] for ex in examples]) if examples else 0
    num_choices = [ex['metadata_num_choices'] for ex in examples]
    avg_choices = np.mean(num_choices) if num_choices else 0
    
    summary_data.append({
        'Dataset': dataset_name,
        'Examples': num_examples,
        'Categories': len(categories),
        'Avg Difficulty': f"{avg_difficulty:.2f}",
        'Avg Contestedness': f"{avg_contestedness:.2f}",
        'Avg Choices': f"{avg_choices:.1f}"
    })

# Print as table
print("\n" + "-"*80)
header = f"{'Dataset':<15} {'Examples':<10} {'Categories':<12} {'Avg Diff':<10} {'Avg Contest':<13} {'Avg Choices':<12}"
print(header)
print("-"*80)

for row in summary_data:
    print(f"{row['Dataset']:<15} {row['Examples']:<10} {row['Categories']:<12} {row['Avg Difficulty']:<10} {row['Avg Contestedness']:<13} {row['Avg Choices']:<12}")

print("-"*80)
print("\nDemo complete! The full dataset contains 3,462 examples across all three datasets.")